In [28]:
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# print("MAE:", mean_absolute_error(y_test, preds))
# print("MSE:", mean_squared_error(y_test, preds))
# print("R2:", r2_score(y_test, preds))


In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from category_encoders.leave_one_out import LeaveOneOutEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_squared_error, r2_score

In [30]:
df = pd.read_csv("VehicalDataSet.csv")
df['price'] = df['price'].fillna(df['price'].median())


In [ ]:

def feature_engineering(df):
    df = df.copy()
    df['car_age'] = 2025 - df['year']
    df['mileage_per_year'] = df['mileage'] / (df['car_age'] + 1)
    return df
df = feature_engineering(df)

In [33]:
target = "price"
X = df.drop(columns=['price','name','description'])
y = df[target]

In [34]:
numeric_features = ["cylinders", "mileage", "doors", "year", "car_age", "mileage_per_year"]
onehot_features = ["fuel", "transmission", "drivetrain", "body"]
loo_features = ["make", "model", "engine", "trim", "exterior_color", "interior_color"]

In [35]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])


In [36]:
onehot_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# 6. Leave-One-Out transformer
loo_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("loo", LeaveOneOutEncoder(handle_unknown="value", handle_missing="value"))
])

In [37]:

X.head()

,make,model,year,engine,cylinders,fuel,mileage,transmission,trim,body,doors,exterior_color,interior_color,drivetrain,car_age,mileage_per_year
0,Jeep,Wagoneer,2024,24V GDI DOHC Twin Turbo,6.0,Gasoline,10.0,8-Speed Automatic,Series II,SUV,4.0,White,Global Black,Four-wheel Drive,1,5.000000
1,Jeep,Grand Cherokee,2024,OHV,6.0,Gasoline,1.0,8-Speed Automatic,Laredo,SUV,4.0,Metallic,Global Black,Four-wheel Drive,1,0.500000
2,GMC,Yukon XL,2024,"6.2L V-8 gasoline direct injection, variable v...",8.0,Gasoline,0.0,Automatic,Denali,SUV,4.0,Summit White,Teak/Light Shale,Four-wheel Drive,1,0.000000
3,Dodge,Durango,2023,16V MPFI OHV,8.0,Gasoline,32.0,8-Speed Automatic,Pursuit,SUV,4.0,White Knuckle Clearcoat,Black,All-wheel Drive,2,10.666667
4,RAM,3500,2024,24V DDI OHV Turbo Diesel,6.0,Diesel,10.0,6-Speed Automatic,Laramie,Pickup Truck,4.0,Silver,Black,Four-wheel Drive,1,5.000000


In [38]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1002 entries, 0 to 1001
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   make              1002 non-null   object 
 1   model             1002 non-null   object 
 2   year              1002 non-null   int64  
 3   engine            1000 non-null   object 
 4   cylinders         897 non-null    float64
 5   fuel              995 non-null    object 
 6   mileage           968 non-null    float64
 7   transmission      1000 non-null   object 
 8   trim              1001 non-null   object 
 9   body              999 non-null    object 
 10  doors             995 non-null    float64
 11  exterior_color    997 non-null    object 
 12  interior_color    964 non-null    object 
 13  drivetrain        1002 non-null   object 
 14  car_age           1002 non-null   int64  
 15  mileage_per_year  968 non-null    float64
dtypes: float64(4), int64(2), object(10)
memory

In [39]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("onehot", onehot_transformer, onehot_features),
        ("loo", loo_transformer, loo_features)
    ],
    remainder="drop"   # drop name, description
)

In [40]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1002 entries, 0 to 1001
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   make              1002 non-null   object 
 1   model             1002 non-null   object 
 2   year              1002 non-null   int64  
 3   engine            1000 non-null   object 
 4   cylinders         897 non-null    float64
 5   fuel              995 non-null    object 
 6   mileage           968 non-null    float64
 7   transmission      1000 non-null   object 
 8   trim              1001 non-null   object 
 9   body              999 non-null    object 
 10  doors             995 non-null    float64
 11  exterior_color    997 non-null    object 
 12  interior_color    964 non-null    object 
 13  drivetrain        1002 non-null   object 
 14  car_age           1002 non-null   int64  
 15  mileage_per_year  968 non-null    float64
dtypes: float64(4), int64(2), object(10)
memory

In [60]:
from xgboost import XGBRegressor
base_rf = XGBRegressor(random_state=42, n_estimators=50)

In [61]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(random_state=42))
])

In [62]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 11. Hyperparameter tuning with GridSearchCV
param_grid = {
    "regressor__n_estimators": [100, 200],
    "regressor__max_depth": [3, 6, 10],
    "regressor__learning_rate": [0.01, 0.1, 0.2],
    "regressor__subsample": [0.6, 0.8, 1.0],
    "regressor__colsample_bytree": [0.6, 0.8, 1.0],
    "regressor__gamma": [0, 1, 5],
    "regressor__reg_alpha": [0, 0.1, 1],
    "regressor__reg_lambda": [1, 1.5, 2]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    n_jobs=-1,
    scoring="r2"
)

#

In [63]:
grid_search.fit(X_train, y_train)

KeyboardInterrupt: 

In [52]:
preds = grid_search.predict(X_test)

In [53]:
r2_score(y_test, preds)

0.8106265064834959

In [45]:
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
# from sklearn.impute import SimpleImputer
# from category_encoders.leave_one_out import LeaveOneOutEncoder
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.feature_selection import RFE
# from sklearn.metrics import mean_squared_error, r2_score

# # 1. Load dataset
# df = pd.read_csv("VehicalDataSet.csv")
# df['price'].fillna(df['price'].median(), inplace=True)

# # ===================== Feature Engineering =====================
# def feature_engineering(df):
#     df = df.copy()
#     df['car_age'] = 2025 - df['year']
#     df['mileage_per_year'] = df['mileage'] / (df['car_age'] + 1)
#     return df

# # Apply feature engineering before split
# df = feature_engineering(df)

# # 2. Define target & features
# target = "price"
# X = df.drop(columns=[target])
# y = df[target]

# # 3. Define feature groups (add engineered features)
# numeric_features = ["cylinders", "mileage", "doors", "year", "car_age", "mileage_per_year"]
# onehot_features = ["fuel", "transmission", "drivetrain", "body"]
# loo_features = ["make", "model", "engine", "trim", "exterior_color", "interior_color"]

# # 4. Numeric transformer
# numeric_transformer = Pipeline(steps=[
#     ("imputer", SimpleImputer(strategy="median"))
# ])

# # 5. One-Hot transformer
# onehot_transformer = Pipeline(steps=[
#     ("imputer", SimpleImputer(strategy="most_frequent")),
#     ("onehot", OneHotEncoder(handle_unknown="ignore"))
# ])

# # 6. Leave-One-Out transformer
# loo_transformer = Pipeline(steps=[
#     ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
#     ("loo", LeaveOneOutEncoder(handle_unknown="value", handle_missing="value"))
# ])

# # 7. Combine all transformations
# preprocessor = ColumnTransformer(
#     transformers=[
#         ("num", numeric_transformer, numeric_features),
#         ("onehot", onehot_transformer, onehot_features),
#         ("loo", loo_transformer, loo_features)
#     ],
#     remainder="drop"   # drop name, description
# )

# # 8. Base regressor for feature selection
# base_rf = RandomForestRegressor(random_state=42, n_estimators=50)

# # 9. Add RFE after preprocessing
# pipeline = Pipeline(steps=[
#     ("preprocessor", preprocessor),
#     ("feature_selector", RFE(base_rf, n_features_to_select=20)),  # keep top 20 features
#     ("regressor", RandomForestRegressor(random_state=42))
# ])

# # 10. Train-test split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # 11. Hyperparameter tuning with GridSearchCV
# param_grid = {
#     "regressor__n_estimators": [100, 200],
#     "regressor__max_depth": [10, 20, None],
#     "regressor__min_samples_split": [2, 5, 10],
#     "regressor__min_samples_leaf": [1, 2, 4]
# }

# grid_search = GridSearchCV(
#     pipeline,
#     param_grid,
#     cv=5,
#     n_jobs=-1,
#     scoring="r2"
# )

# grid_search.fit(X_train, y_train)

# # 12. Best model
# best_model = grid_search.best_estimator_
# print("Best Parameters:", grid_search.best_params_)

# # 13. Evaluate
# y_pred = best_model.predict(X_test)
# print("R2 Score:", r2_score(y_test, y_pred))
# print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
